In [11]:
# -*- coding: utf-8 -*-
"""
QPT Hardware Pipeline for the Deterministic DCTC Circuit
=========================================================
Characterises the channel Phi: rho_M -> output on C
by running 12 circuits (4 probe messages x 3 Pauli bases)
on ibm_kingston via SamplerV2 Batch with XX dynamical decoupling.

From the reconstructed PTM we compute:
  - Entanglement fidelity  F_e = (1 + Tr(T)) / 4
  - Process infidelity     r   = 1 - F_e
  - Diamond norm bound     delta = ||Phi_noisy - id||_diamond
  - Certified fixed-point  ||sigma* - rho_M||_1 <= delta  [Lemma II.2]

Bootstrap (N_bs=2000) gives 95% CIs on all quantities.

Usage:
  python qpt_hardware_pipeline.py simulator   # noiseless verification
  python qpt_hardware_pipeline.py hardware    # run on ibm_kingston
  python qpt_hardware_pipeline.py postprocess # re-process saved raw results
"""

import numpy as np
import json
import sys
from math import pi
from pathlib import Path


from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import XGate

# ── Pauli matrices ─────────────────────────────────────────────────────────────
I2 = np.eye(2, dtype=complex)
sx = np.array([[0,1],[1,0]], dtype=complex)
sy = np.array([[0,-1j],[1j,0]], dtype=complex)
sz = np.array([[1,0],[0,-1]], dtype=complex)

def bloch(rho):
    return np.real([np.trace(rho@sx), np.trace(rho@sy), np.trace(rho@sz)])

def ideal_rho(th, ph):
    qc = QuantumCircuit(1); qc.u(th, ph, 0.0, 0)
    sv = Statevector(qc)
    return np.outer(sv.data, sv.data.conj())

# ── Experimental parameters ────────────────────────────────────────────────────
BACKEND_NAME    = "ibm_kingston"
INITIAL_LAYOUT  = None   # let transpiler choose best layout for kingston's 156-qubit heavy-hex
OPT_LEVEL       = 3
SEED            = 42
NSHOTS          = 10_000
N_BOOTSTRAP     = 2000
SAVE_PATH       = Path("qpt_raw_results.json")
RESULTS_PATH    = Path("qpt_final_results.json")

# ── QPT probe states: 4 linearly independent messages ─────────────────────────
PROBE_STATES = {
    '|0>':   (0,      0),
    '|1>':   (pi,     0),
    '|+>':   (pi/2,   0),
    '|+i>':  (pi/2,   pi/2),
}
BASES = ['Z', 'X', 'Y']   # Pauli measurement bases on C


# ══════════════════════════════════════════════════════════════════════════════
# 1. CIRCUIT BUILDER
# ══════════════════════════════════════════════════════════════════════════════

def build_qpt_circuit(theta_m, phi_m, basis):
    """
    Deterministic DCTC circuit for QPT.

    Steps:
      1.  U(theta_m, phi_m, 0) on M  — prepare message rho_M
      2.  SWAP(C,M)                  — C = rho_M, M = |0>
      3.  Bell pairs: (E,M),(R,G),(A,Y)
      4.  Scrambler U(C,E,R)
      5.  Decoder U†(G,M,A)
      6.  Grover(R,G)
      7.  Decoder U^T(G,M,A)
      8.  Grover(A,Y)
      9.  Decoder U†(G,M,A)
      10. SWAP(C,Y)                  — close CTC loop, output on C
      11. Pauli basis rotation on C  — for QPT measurement
      12. Measure C

    No mid-circuit measurements. No post-selection.
    """
    C = QuantumRegister(1,'C'); E = QuantumRegister(1,'E')
    R = QuantumRegister(1,'R'); G = QuantumRegister(1,'G')
    M = QuantumRegister(1,'M'); A = QuantumRegister(1,'A')
    Y = QuantumRegister(1,'Y')
    crC = ClassicalRegister(1,'crC')
    qc  = QuantumCircuit(C,E,R,G,M,A,Y,crC)

    # Step 1-2: message into C
    qc.u(theta_m, phi_m, 0.0, M[0])
    qc.swap(C, M)
    # Step 3: Bell pairs
    qc.h(E[0]); qc.cx(E[0], M[0])
    qc.h(R[0]); qc.cx(R[0], G[0])
    qc.h(A[0]); qc.cx(A[0], Y[0])
    # Step 4: Scrambler U(C,E,R)
    qc.cz(C[0],R[0]); qc.cz(E[0],R[0]); qc.cz(C[0],E[0])
    qc.h(C[0]); qc.h(E[0]); qc.h(R[0])
    qc.cz(C[0],R[0]); qc.cz(C[0],E[0]); qc.cz(E[0],R[0])
    # Step 5: Decoder U†(G,M,A)
    qc.cz(A[0],G[0]); qc.cz(M[0],A[0]); qc.cz(G[0],M[0])
    qc.h(A[0]); qc.h(M[0]); qc.h(G[0])
    qc.cz(A[0],G[0]); qc.cz(G[0],M[0]); qc.cz(M[0],A[0])
    # Step 6: Grover(R,G)
    qc.rz(pi,R[0]); qc.rx(pi,R[0]); qc.rx(pi,G[0])
    qc.swap(R[0],G[0]); qc.rz(pi,R[0])
    # Step 7: Decoder U^T(G,M,A)
    qc.cz(A[0],G[0]); qc.cz(M[0],A[0]); qc.cz(G[0],M[0])
    qc.h(A[0]); qc.h(M[0]); qc.h(G[0])
    qc.cz(A[0],G[0]); qc.cz(G[0],M[0]); qc.cz(M[0],A[0])
    # Step 8: Grover(A,Y)
    qc.rz(pi,A[0]); qc.rx(pi,A[0]); qc.rx(pi,Y[0])
    qc.swap(A[0],Y[0]); qc.rz(pi,A[0])
    # Step 9: Decoder U†(G,M,A) again
    qc.cz(A[0],G[0]); qc.cz(M[0],A[0]); qc.cz(G[0],M[0])
    qc.h(A[0]); qc.h(M[0]); qc.h(G[0])
    qc.cz(A[0],G[0]); qc.cz(G[0],M[0]); qc.cz(M[0],A[0])
    # Step 10: SWAP(C,Y) — close CTC loop
    qc.swap(C, Y)
    # Step 11: Pauli basis rotation on C
    if basis == 'X':
        qc.h(C[0])
    elif basis == 'Y':
        qc.sdg(C[0]); qc.h(C[0])
    # Step 12: measure C
    qc.measure(C[0], crC[0])
    return qc


# ══════════════════════════════════════════════════════════════════════════════
# 2. TRANSPILE WITH XX DYNAMICAL DECOUPLING
# ══════════════════════════════════════════════════════════════════════════════

def transpile_with_dd(circuits, backend):
    """Transpile at opt level 3 with XX DD sequences."""
    transpiled = transpile(
        circuits,
        backend=backend,
        optimization_level=OPT_LEVEL,
        initial_layout=INITIAL_LAYOUT,   # None = let transpiler choose best layout
        seed_transpiler=SEED,
    )
    try:
        from qiskit_ibm_runtime.transpiler.passes.scheduling import (
            ALAPScheduleAnalysis, PadDynamicalDecoupling
        )
        from qiskit.transpiler import PassManager, InstructionDurations
        durations = InstructionDurations.from_backend(backend)
        dd_pm = PassManager([
            ALAPScheduleAnalysis(durations),
            PadDynamicalDecoupling(durations, [XGate(), XGate()]),
        ])
        transpiled = dd_pm.run(transpiled)
        print("  XX dynamical decoupling applied.")
    except Exception as e:
        print(f"  DD not applied ({e}), proceeding without.")
    return transpiled


# ══════════════════════════════════════════════════════════════════════════════
# 3. HARDWARE EXECUTION
# ══════════════════════════════════════════════════════════════════════════════

def run_hardware():
    """Submit all 12 QPT circuits in a single Batch job on ibm_kingston."""
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2, Batch

    service = QiskitRuntimeService(instance='CTCs')
    backend = service.backend(BACKEND_NAME)
    print(f"Backend: {BACKEND_NAME}")

    # Build all 12 circuits
    labels, circuits = [], []
    for msg_label, (th, ph) in PROBE_STATES.items():
        for basis in BASES:
            labels.append((msg_label, basis))
            circuits.append(build_qpt_circuit(th, ph, basis))
    print(f"Built {len(circuits)} QPT circuits.")

    # Transpile
    print("Transpiling...")
    transpiled = transpile_with_dd(circuits, backend)
    depths = [tc.depth() for tc in transpiled]
    twoq  = [sum(1 for g in tc.data if g.operation.num_qubits==2)
             for tc in transpiled]
    print(f"  Depths: min={min(depths)}, max={max(depths)}")
    print(f"  2Q gates: min={min(twoq)}, max={max(twoq)}")

    # Submit in one Batch
    raw_results = {}
    with Batch(backend=backend) as batch:
        sampler = SamplerV2(mode=batch)
        job = sampler.run(transpiled, shots=NSHOTS)
        job_id = job.job_id()
        print(f"  Job submitted: {job_id}")
        # Save job ID immediately in case connection drops
        with open("qpt_job_id.txt", "w") as f:
            f.write(job_id)
        print(f"  Job ID saved to qpt_job_id.txt")
        print(f"  Waiting for results...")
        result = job.result()
        print(f"  Job complete.")

    for i, (msg_label, basis) in enumerate(labels):
        bits = result[i].data.crC.get_bitstrings()
        counts = {}
        for b in bits:
            counts[b] = counts.get(b, 0) + 1
        raw_results[str((msg_label, basis))] = counts
        n0 = counts.get('0', 0); n1 = counts.get('1', 0)
        print(f"  {msg_label}, {basis}: n0={n0}, n1={n1}, "
              f"<sigma>={( n0-n1)/NSHOTS:+.4f}")

    with open(SAVE_PATH, 'w') as f:
        json.dump(raw_results, f, indent=2)
    print(f"Raw results saved to {SAVE_PATH}")
    return raw_results


# ══════════════════════════════════════════════════════════════════════════════
# 4. SIMULATOR EXECUTION
# ══════════════════════════════════════════════════════════════════════════════

def run_simulator():
    """Run all 12 QPT circuits on the noiseless AerSimulator."""
    from qiskit_aer import AerSimulator
    sim = AerSimulator()
    raw_results = {}
    for msg_label, (th, ph) in PROBE_STATES.items():
        for basis in BASES:
            qc = build_qpt_circuit(th, ph, basis)
            counts = sim.run(
                transpile(qc, sim), shots=NSHOTS
            ).result().get_counts()
            raw_results[str((msg_label, basis))] = counts
            n0 = counts.get('0',0); n1 = counts.get('1',0)
            print(f"  {msg_label}, {basis}: n0={n0}, n1={n1}, "
                  f"<sigma>={( n0-n1)/NSHOTS:+.4f}")
    with open(SAVE_PATH, 'w') as f:
        json.dump(raw_results, f, indent=2)
    print(f"Saved to {SAVE_PATH}")
    return raw_results


# ══════════════════════════════════════════════════════════════════════════════
# 5. QPT POST-PROCESSING
# ══════════════════════════════════════════════════════════════════════════════

def ev(raw, msg_label, basis):
    """Pauli expectation value from counts."""
    key = str((msg_label, basis))
    counts = raw[key]
    n0 = counts.get('0', 0); n1 = counts.get('1', 0)
    total = n0 + n1
    return (n0 - n1) / total if total > 0 else 0.0

def reconstruct_rho(raw, msg_label):
    """Reconstruct output density matrix from 3 Pauli measurements."""
    ex = ev(raw, msg_label, 'X')
    ey = ev(raw, msg_label, 'Y')
    ez = ev(raw, msg_label, 'Z')
    rho = 0.5*(I2 + ex*sx + ey*sy + ez*sz)
    w, v = np.linalg.eigh(rho)
    w = np.clip(w, 0, None); w /= w.sum()
    return v @ np.diag(w) @ v.conj().T

def compute_ptm(raw):
    """
    Compute the affine Bloch map of Phi:
      r_out = t + T @ r_in
    from 4 probe states.
    Returns t (3,) and T (3,3).
    """
    labels = list(PROBE_STATES.keys())
    rho_in_dict  = {l: ideal_rho(th,ph) for l,(th,ph) in PROBE_STATES.items()}
    rho_out_dict = {l: reconstruct_rho(raw,l) for l in labels}

    B_in  = np.column_stack([[1]+list(bloch(rho_in_dict[l]))  for l in labels])
    B_out = np.column_stack([[1]+list(bloch(rho_out_dict[l])) for l in labels])
    A = B_out @ np.linalg.inv(B_in)
    return A[1:,0], A[1:,1:], rho_out_dict   # t, T, rho_out

def compute_delta(t, T):
    """
    Diamond norm ||Phi_noisy - id||_diamond via analytical formula.

    The error map E = Phi - id has Bloch action:
      E(r) = t + (T - I3) @ r

    max_{|r|<=1} ||t + T_err @ r||
    = ||t|| + sigma_max(T_err)      [triangle inequality, tight when aligned]

    where sigma_max is the largest singular value of T_err = T - I3.
    This is an upper bound; for the paper it serves as a valid certified bound.
    """
    T_err = T - np.eye(3)
    sigma_max = float(np.linalg.svd(T_err, compute_uv=False)[0])
    t_norm    = float(np.linalg.norm(t))
    return t_norm + sigma_max

def compute_all(raw):
    """Full QPT: PTM -> F_e -> delta -> certified bound."""
    t, T, rho_out = compute_ptm(raw)
    Fe    = float(np.clip((1.0 + np.real(np.trace(T))) / 4.0, 0, 1))
    r     = 1.0 - Fe
    delta = compute_delta(t, T)
    return {
        't': t.tolist(),
        'T': T.tolist(),
        'Fe': Fe,
        'r': r,
        'delta': delta,
        'certified_bound': delta,
        'rho_out': {k: v.tolist() for k,v in rho_out.items()},
    }


# ══════════════════════════════════════════════════════════════════════════════
# 6. BOOTSTRAP CONFIDENCE INTERVALS
# ══════════════════════════════════════════════════════════════════════════════

def bootstrap_qpt(raw, n_bs=N_BOOTSTRAP):
    """
    Non-parametric bootstrap on QPT counts.
    For each resample: redraw multinomial counts, recompute Fe and delta.
    Returns 95% CIs on Fe, r, delta.
    """
    print(f"\nBootstrap ({n_bs} resamples)...")
    Fe_bs    = []
    delta_bs = []
    rng = np.random.default_rng(0)

    for b in range(n_bs):
        raw_bs = {}
        for key, counts in raw.items():
            n0 = counts.get('0',0); n1 = counts.get('1',0)
            total = n0 + n1
            if total == 0:
                raw_bs[key] = counts; continue
            p0 = n0 / total
            new_n0 = int(rng.binomial(total, p0))
            raw_bs[key] = {'0': new_n0, '1': total - new_n0}

        t_b, T_b, _ = compute_ptm(raw_bs)
        Fe_b    = float(np.clip((1 + np.real(np.trace(T_b))) / 4, 0, 1))
        delta_b = compute_delta(t_b, T_b)
        Fe_bs.append(Fe_b)
        delta_bs.append(delta_b)

        if (b+1) % 200 == 0:
            print(f"  {b+1}/{n_bs} resamples done...")

    Fe_bs    = np.array(Fe_bs)
    delta_bs = np.array(delta_bs)
    return {
        'Fe_mean':    float(np.mean(Fe_bs)),
        'Fe_ci':      np.percentile(Fe_bs, [2.5, 97.5]).tolist(),
        'r_mean':     float(np.mean(1-Fe_bs)),
        'r_ci':       np.percentile(1-Fe_bs, [2.5, 97.5]).tolist(),
        'delta_mean': float(np.mean(delta_bs)),
        'delta_ci':   np.percentile(delta_bs, [2.5, 97.5]).tolist(),
        'delta_upper_95': float(np.percentile(delta_bs, 97.5)),
    }


# ══════════════════════════════════════════════════════════════════════════════
# 7. PRINT REPORT
# ══════════════════════════════════════════════════════════════════════════════

def print_report(results, bs):
    print()
    print("=" * 60)
    print("  QPT RESULTS — LEMMA II.2 CERTIFIED BOUND")
    print("=" * 60)
    print()
    print(f"  Entanglement fidelity  F_e = {results['Fe']:.6f}")
    print(f"    95% CI: [{bs['Fe_ci'][0]:.6f}, {bs['Fe_ci'][1]:.6f}]")
    print()
    print(f"  Process infidelity     r   = {results['r']:.6f}")
    print(f"    95% CI: [{bs['r_ci'][0]:.6f}, {bs['r_ci'][1]:.6f}]")
    print()
    print(f"  Diamond norm bound   delta = {results['delta']:.6f}")
    print(f"    95% CI: [{bs['delta_ci'][0]:.6f}, {bs['delta_ci'][1]:.6f}]")
    print()
    print(f"  Lemma II.2 certified bound (95% upper):")
    print(f"    ||sigma* - rho_M||_1 <= {bs['delta_upper_95']:.6f}")
    print()
    print("  Contraction matrix T (diagonal):")
    T = np.array(results['T'])
    print(f"    T_xx = {T[0,0]:.6f}  (ideal: 1)")
    print(f"    T_yy = {T[1,1]:.6f}  (ideal: 1)")
    print(f"    T_zz = {T[2,2]:.6f}  (ideal: 1)")
    print()
    print("  Translation t:")
    t = results['t']
    print(f"    t = [{t[0]:.6f}, {t[1]:.6f}, {t[2]:.6f}]  (ideal: [0,0,0])")
    print()
    print("  Output fidelities F(Phi(rho_M), rho_M) per probe:")
    for label, (th, ph) in PROBE_STATES.items():
        rho_M_ideal = ideal_rho(th, ph)
        rho_M_out   = np.array(results['rho_out'][label])
        sv = Statevector(QuantumCircuit(1))
        qc_msg = QuantumCircuit(1); qc_msg.u(th,ph,0,0)
        sv_msg = Statevector(qc_msg)
        f = float(np.real(sv_msg.data.conj() @ rho_M_out @ sv_msg.data))
        print(f"    {label}: F = {f:.6f}")
    print("=" * 60)


# ══════════════════════════════════════════════════════════════════════════════
# 8. MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    # Find the mode argument — ignore Jupyter kernel args (start with '--' or '-f')
    mode = "simulator"   # default
    for arg in sys.argv[1:]:
        if arg in ("simulator", "hardware", "postprocess", "retrieve"):
            mode = arg
            break

    if mode == "simulator":
        print("=== Noiseless simulator QPT ===")
        raw = run_simulator()

    elif mode == "hardware":
        print(f"=== Hardware QPT on {BACKEND_NAME} ===")
        raw = run_hardware()

    elif mode == "retrieve":
        # Fetch results from a previously submitted job
        job_id_file = Path("qpt_job_id.txt")
        if not job_id_file.exists():
            print("No qpt_job_id.txt found. Run hardware mode first.")
            sys.exit(1)
        job_id = job_id_file.read_text().strip()
        print(f"Retrieving job {job_id} from {BACKEND_NAME}...")
        from qiskit_ibm_runtime import QiskitRuntimeService
        service = QiskitRuntimeService(instance='CTCs')
        job = service.job(job_id)
        result = job.result()
        labels = [(msg, basis)
                  for msg in PROBE_STATES
                  for basis in BASES]
        raw = {}
        for i, (msg_label, basis) in enumerate(labels):
            bits = result[i].data.crC.get_bitstrings()
            counts = {}
            for b in bits:
                counts[b] = counts.get(b, 0) + 1
            raw[str((msg_label, basis))] = counts
        with open(SAVE_PATH, 'w') as f:
            json.dump(raw, f, indent=2)
        print(f"Results saved to {SAVE_PATH}")

    elif mode == "postprocess":
        print(f"=== Post-processing saved results from {SAVE_PATH} ===")
        with open(SAVE_PATH) as f:
            raw = json.load(f)

    print("\nComputing QPT results...")
    results = compute_all(raw)

    print("Computing bootstrap CIs...")
    bs = bootstrap_qpt(raw)

    print_report(results, bs)

    # Save final results — convert all numpy/complex types to plain Python
    def to_python(obj):
        if isinstance(obj, dict):
            return {k: to_python(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [to_python(v) for v in obj]
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, (np.integer,)):
            return int(obj)
        elif isinstance(obj, (np.floating,)):
            return float(obj)
        elif isinstance(obj, (np.complexfloating, complex)):
            return [float(obj.real), float(obj.imag)]
        else:
            return obj

    final = to_python({'point_estimates': results, 'bootstrap': bs})
    with open(RESULTS_PATH, 'w') as f:
        json.dump(final, f, indent=2)
    print(f"\nFull results saved to {RESULTS_PATH}")

=== Noiseless simulator QPT ===
  |0>, Z: n0=10000, n1=0, <sigma>=+1.0000
  |0>, X: n0=5036, n1=4964, <sigma>=+0.0072
  |0>, Y: n0=4986, n1=5014, <sigma>=-0.0028
  |1>, Z: n0=0, n1=10000, <sigma>=-1.0000
  |1>, X: n0=5047, n1=4953, <sigma>=+0.0094
  |1>, Y: n0=5006, n1=4994, <sigma>=+0.0012
  |+>, Z: n0=5020, n1=4980, <sigma>=+0.0040
  |+>, X: n0=10000, n1=0, <sigma>=+1.0000
  |+>, Y: n0=4980, n1=5020, <sigma>=-0.0040
  |+i>, Z: n0=5084, n1=4916, <sigma>=+0.0168
  |+i>, X: n0=4947, n1=5053, <sigma>=-0.0106
  |+i>, Y: n0=10000, n1=0, <sigma>=+1.0000
Saved to qpt_raw_results.json

Computing QPT results...
Computing bootstrap CIs...

Bootstrap (2000 resamples)...
  200/2000 resamples done...
  400/2000 resamples done...
  600/2000 resamples done...
  800/2000 resamples done...
  1000/2000 resamples done...
  1200/2000 resamples done...
  1400/2000 resamples done...
  1600/2000 resamples done...
  1800/2000 resamples done...
  2000/2000 resamples done...

  QPT RESULTS — LEMMA II.2 CERTIFI

In [15]:
raw     = run_hardware()     # submits to ibm_torino, waits for result
results = compute_all(raw)
bs      = bootstrap_qpt(raw)
print_report(results, bs)

Backend: ibm_kingston
Built 12 QPT circuits.
Transpiling...
  XX dynamical decoupling applied.
  Depths: min=85, max=87
  2Q gates: min=48, max=48
  Job submitted: d7h8pp22khts739pffi0
  Job ID saved to qpt_job_id.txt
  Waiting for results...
  Job complete.
  |0>, Z: n0=9155, n1=845, <sigma>=+0.8310
  |0>, X: n0=4923, n1=5077, <sigma>=-0.0154
  |0>, Y: n0=4949, n1=5051, <sigma>=-0.0102
  |1>, Z: n0=768, n1=9232, <sigma>=-0.8464
  |1>, X: n0=4960, n1=5040, <sigma>=-0.0080
  |1>, Y: n0=4951, n1=5049, <sigma>=-0.0098
  |+>, Z: n0=4964, n1=5036, <sigma>=-0.0072
  |+>, X: n0=8982, n1=1018, <sigma>=+0.7964
  |+>, Y: n0=4502, n1=5498, <sigma>=-0.0996
  |+i>, Z: n0=5248, n1=4752, <sigma>=+0.0496
  |+i>, X: n0=5502, n1=4498, <sigma>=+0.1004
  |+i>, Y: n0=9055, n1=945, <sigma>=+0.8110
Raw results saved to qpt_raw_results.json

Bootstrap (2000 resamples)...
  200/2000 resamples done...
  400/2000 resamples done...
  600/2000 resamples done...
  800/2000 resamples done...
  1000/2000 resamples do